#### torch.Tensor
A torch.Tensor is a multi-dimensional matrix containing elements of a single data type. Please see torch.dtype for more details about dtype support.

***Initializing and basic operations***
A tensor can be constructed from a Python list or sequence using the torch.tensor() constructor:

In [18]:
import torch 
t1 = torch.tensor([[1., -1.], [1., -1.]])
t1

tensor([[ 1., -1.],
        [ 1., -1.]])

In [19]:
import numpy as np
t2 = torch.tensor(np.array([ [1,3,5],[2,4,6]]))
t2

tensor([[1, 3, 5],
        [2, 4, 6]])

In [22]:
t3 = torch.tensor([10.0, 20.0, 30.0] , requires_grad=True)
print(t3)
#sets requires_grad=True in-place
print(t3.requires_grad)

tensor([10., 20., 30.], requires_grad=True)
True


In [29]:
predicted_y = t3.sum()
print(f"Initial tensor:{t3}")
print(f"Afer sum() operation: {predicted_y}")
print("backpropagation!")
predicted_y.backward()

print(t3.grad)


Initial tensor:tensor([10., 20., 30.], requires_grad=True)
Afer sum() operation: 60.0
backpropagation!
tensor([3., 3., 3.])


In [26]:
import torch
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [27]:
x.requires_grad_()  # sets requires_grad=True in-place


tensor([1., 2., 3.], requires_grad=True)

In [28]:
y = x.sum()
y.backward()
print(x.grad)  # tensor([1., 1., 1.])

tensor([1., 1., 1.])


#### PyTorch detach() Usage

The torch.Tensor.detach() method creates a new tensor that shares the same storage as the original but is detached from the computational graph, meaning it will not track gradients. This is useful when you want to prevent certain tensors from contributing to gradient computation, saving memory and computation time.

In [30]:
import torch

# Create a tensor with gradient tracking
x = torch.tensor([2.0, 3.0], requires_grad=True)

# Perform an operation
y = x * 5

# Detach y from the computation graph
y_detached = y.detach()

print("Original:", y) # requires_grad=True
print("Detached:", y_detached) # requires_grad=False

Original: tensor([10., 15.], grad_fn=<MulBackward0>)
Detached: tensor([10., 15.])


#### Key Points:

* Purpose: Returns a tensor that will never require gradients and is not part of the autograd graph.

* Storage Sharing: The detached tensor shares memory with the original, so in-place changes affect both.

Syntax:
```python
detached_tensor = tensor.detach()
```

Common Use Cases:
* Reduce Memory Usage – Avoid storing intermediate gradients when they are not needed.

```python
features = model.encoder(data).detach()
output = model.decoder(features)
```

* Speed Up Computation – Skip gradient tracking for parts of the model not being trained.

* Stabilize Training – Isolate layers prone to exploding/vanishing gradients by detaching their outputs.

***Important Considerations:***

* Detached tensors cannot be used for backpropagation beyond their creation point.

* Over-detaching can block necessary gradient flow, harming training.

* For disabling gradients globally in a block, consider torch.no_grad() or @torch.inference_mode instead of multiple .detach() calls.

* In short, use .detach() when you need a tensor’s value but don’t want it to be part of gradient computation — especially in scenarios involving feature extraction, inference, or memory optimization.

#### A tensor of specific data type can be constructed by passing a torch.dtype and/or a torch.device to a constructor or tensor creation op:


In [31]:
torch.zeros([2, 4], dtype=torch.int32)

tensor([[0, 0, 0, 0],
        [0, 0, 0, 0]], dtype=torch.int32)

In [32]:
cuda0 = torch.device('cuda:0')
torch.ones([2, 4], dtype=torch.float64, device=cuda0)
'''
tensor([[ 1.0000,  1.0000,  1.0000,  1.0000],
        [ 1.0000,  1.0000,  1.0000,  1.0000]], dtype=torch.float64, device='cuda:0')
'''

AssertionError: Torch not compiled with CUDA enabled

The contents of a tensor can be accessed and modified using Python’s indexing and slicing notation:

In [33]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(x[1][2])

tensor(6)


In [34]:
x[0][1] = 8
print(x)

tensor([[1, 8, 3],
        [4, 5, 6]])


In [42]:
x1 = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=float, requires_grad=False)
print(x1)
x1[0][1] = 10
print(x1)


tensor([[1., 2., 3.],
        [4., 5., 6.]], dtype=torch.float64)
tensor([[ 1., 10.,  3.],
        [ 4.,  5.,  6.]], dtype=torch.float64)


In [45]:
x = torch.tensor([[1., -1.], [1., 1.]], requires_grad=True)
out = x.pow(2).sum()
out.backward()
x.grad

tensor([[ 2., -2.],
        [ 2.,  2.]])

AttributeError: 'int' object has no attribute 'pow'

CUDA semantics
Created On: Jan 16, 2017 | Last Updated On: Sep 05, 2026

torch.cuda is used to set up and run CUDA operations. It keeps track of the currently selected GPU, and all CUDA tensors you allocate will by default be created on that device. The selected device can be changed with a torch.cuda.device context manager.

However, once a tensor is allocated, you can do operations on it irrespective of the selected device, and the results will be always placed on the same device as the tensor.

Cross-GPU operations are not allowed by default, with the exception of copy_() and other methods with copy-like functionality such as to() and cuda(). Unless you enable peer-to-peer memory access, any attempts to launch ops on tensors spread across different devices will raise an error.

In [48]:
cuda = torch.device('cuda')     # Default CUDA device
cuda0 = torch.device('cuda:0')
cuda2 = torch.device('cuda:2')  # GPU 2 (these are 0-indexed)

x = torch.tensor([1., 2.], device=cuda0)
# x.device is device(type='cuda', index=0)
y = torch.tensor([1., 2.]).cuda()
# y.device is device(type='cuda', index=0)

with torch.cuda.device(1):
    # allocates a tensor on GPU 1
    a = torch.tensor([1., 2.], device=cuda)

    # transfers a tensor from CPU to GPU 1
    b = torch.tensor([1., 2.]).cuda()
    # a.device and b.device are device(type='cuda', index=1)

    # You can also use ``Tensor.to`` to transfer a tensor:
    b2 = torch.tensor([1., 2.]).to(device=cuda)
    # b.device and b2.device are device(type='cuda', index=1)

    c = a + b
    # c.device is device(type='cuda', index=1)

    z = x + y
    # z.device is device(type='cuda', index=0)

    # even within a context, you can specify the device
    # (or give a GPU index to the .cuda call)
    d = torch.randn(2, device=cuda2)
    e = torch.randn(2).to(cuda2)
    f = torch.randn(2).cuda(cuda2)
    # d.device, e.device, and f.device are all device(type='cuda', index=2)

AssertionError: Torch not compiled with CUDA enabled